# Normalizing flows 

Nessa atividade de laboratório vamos utilizar um modelo semelhante ao Real NVP

DINH, Laurent; SOHL-DICKSTEIN, Jascha; BENGIO, Samy. Density estimation using real nvp. arXiv preprint arXiv:1605.08803, 2016.


In [ ]:
from torch import distributions
import torch
import torch.nn as nn
import numpy as np

import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn import datasets
from sklearn.preprocessing import StandardScaler

torch.backends.cudnn.benchmark = True
device = 'cuda' if torch.cuda.is_available()  else 'cpu'

In [ ]:
n_samples = 1000

# Define distribution.
noisy_moons = datasets.make_moons(n_samples=n_samples, noise=.05)
X, y = noisy_moons
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Plot.
xlim, ylim = [-2, 2], [-2, 2]
plt.scatter(X[:, 0], X[:, 1], s=10, color='red')
plt.xlim(xlim)
plt.title('Noisy two moons distribution')
plt.ylim(ylim);

In [ ]:
def sample_n01(N):
    return np.random.normal(size = (N, 2))

X_normal = sample_n01(1000)

# TODO plotar tanto o two moons quanto a normal

plt.show()

### Loglikelihood do two moons contra uma distribuição normal
Os dados do dataset twomoons são bimodais. Isso é evidente visto que o conjunto de dados possui duas luas.

In [ ]:
# avalia a log verossimilhança dos dados em relação a normal
def log_prob_n01(x):
    return np.sum(- np.square(x) / 2 - np.log(np.sqrt(2 * np.pi)), axis=-1)

plt.hist(log_prob_n01(X), bins=50)
plt.show()

### Para comparação, olhem como seria avaliar a log verossimilhança de dados que seguem uma normal

In [ ]:
plt.hist(log_prob_n01(X_normal), bins=50)
plt.show()

## Vamos definir a classe que implementa o Normalizing Flow
O modelo abaixo implementa uma transformação invertível que um dos componentes é um MLP.
Repare que a transformação não é realizada diretamente pelo MLP, mas sim por uma transformação linear cujos parâmetros são definidos pela saída do MLP.

In [ ]:
class NVP(nn.Module):
    def __init__(self, flips, D=2):
        super().__init__()
        self.D = D
        self.flips = flips
        self.prior = distributions.MultivariateNormal(torch.zeros(2), torch.eye(2))
        self.shift_log_scale_fns = nn.ModuleList()
        for _ in flips:
            shift_log_scale_fn = nn.Sequential(
                nn.Linear(1, 256),
                nn.ReLU(),
                nn.Linear(256, 256),
                nn.ReLU(),
                nn.Linear(256, D),
            )
            self.shift_log_scale_fns.append(shift_log_scale_fn)

    def forward(self, x, flip_idx):
        # x is of shape [B, H]
        flip = self.flips[flip_idx]
        d = x.shape[-1] // 2
        x1, x2 = x[:, :d], x[:, d:]
        if flip:
            x2, x1 = x1, x2
        net_out = self.shift_log_scale_fns[flip_idx](x1)
        shift = net_out[:, :self.D // 2]
        log_scale = net_out[:, self.D // 2:]
        y2 = x2 * torch.exp(log_scale) + shift
        if flip:
            x1, y2 = y2, x1
        y = torch.cat([x1, y2], -1)
        return y

    def inverse_forward(self, y, flip_idx):
        flip = self.flips[flip_idx]
        d = y.shape[-1] // 2
        y1, y2 = y[:, :d], y[:, d:]
        if flip:
            y1, y2 = y2, y1
        net_out = self.shift_log_scale_fns[flip_idx](y1)
        shift = net_out[:, :self.D // 2]
        log_scale = net_out[:, self.D // 2:]
        x2 = (y2 - shift) * torch.exp(-log_scale)
        if flip:
            y1, x2 = x2, y1
        x = torch.cat([y1, x2], -1)
        return x, log_scale

    @staticmethod
    def base_log_prob_fn(x):
        return torch.sum(- (x ** 2) / 2 - np.log(np.sqrt(2 * np.pi)), -1)

    def base_sample_fn(self, N):
        # sampler random normal(0, I)
        x = self.prior.sample((N, 1)).to(device).squeeze(1)
        return x

    def log_prob(self, y, flip_idx):
        x, log_scale = self.inverse_forward(y, flip_idx)
        # This comes from the jacobian. In this case the jacobian is simply the product of the scales,
        # which becomes the sum of log scales in the loglikelihood.
        ildj = - torch.sum(log_scale, -1)
        return self.base_log_prob_fn(x) + ildj

    def sample_nvp_chain(self, N):
        xs = []
        x = self.base_sample_fn(N)
        xs.append(x)
        for i, _ in enumerate(self.flips):
            x = self.forward(x, flip_idx=i)
            xs.append(x)
        return x, xs

    def log_prob_chain(self, y):
        # Run y through all the necessary inverses, keeping track
        # of the logscale along the way, allowing us to compute the loss.
        temp = y
        logscales = y.data.new(y.shape[0]).zero_()
        for i, _ in enumerate(self.flips):
            temp, logscale = self.inverse_forward(
                temp,
                flip_idx=len(self.flips) - 1 - i,
            )
            # One logscale per element in a batch per layer of flow.
            logscales += logscale.squeeze(-1)
        return self.base_log_prob_fn(temp) - logscales

### Definir o modelo NVP.
Aqui vamos instanciar a rede neural NVP. Para criar essa rede, definimos o número de camadas e em quais delas ocorrerá um flip da ordem de mudança dos dados (lembrem que em *coupling flows* precisamos alternar as posições dos splits dos vetores).

In [ ]:
flips = [False, True, False, True, False, True]

# Feel free to try different flips lists. Eg:

#  (More lightweight model, but less model capacity. Does not work very well.)
# flips = [False, True]

#  (Intermediary model. Works decently but fails to model the noisy moons properly.)
# flips = [False, True, False, True]

#  (Heavier model. Works well but is quite heavy. Might overload the GPU.)
# flips = [False, True, False, True, False, True, False, True]

model = NVP(flips).to(device)

In [ ]:
loglikelihoods = model.log_prob_chain(torch.FloatTensor(X).to(device)).data.cpu().numpy()
plt.hist(loglikelihoods)
plt.show()

## Loop de treinamento

In [ ]:
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

iters = 5000

In [ ]:

min_loss = float('inf')

for i in range(iters):
    # Obtemos dados do nosso batch de treinamento
    noisy_moons = datasets.make_moons(n_samples=128, noise=.05)[0].astype(np.float32)
    X = scaler.transform(noisy_moons)

    optimizer.zero_grad()

    batch = torch.FloatTensor(noisy_moons).to(device)
    out_forwardpass = model.log_prob_chain(batch)
    loss = - torch.mean(out_forwardpass)
    if loss.item() < min_loss:
        bestmodel = model

    loss.backward()
    optimizer.step()
    if i % 500 == 0:
        print('Iter {}, loss is {:.3f}'.format(i, loss.item()))

### Com o modelo treinado, agora vamos realizar amostragens.
Vamos amostrar de uma normal $N(0, I)$, e aplicar a sequência de transformações aprendidas.

In [ ]:
new_Xs, _ = bestmodel.sample_nvp_chain(1000)
new_Xs = new_Xs.data.cpu().numpy()

plt.scatter(new_Xs[:, 0], new_Xs[:, 1], c='r', s=1)
plt.show()

## Atividade
Realize plot de cada uma das etapas da transformação aplicada a distribuição normal. Você vai ver as instâncias de treinamento se ajustando as novas posições